# Fine-tune qwen2.5:3b for the data analyst agent (free Colab T4)

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells in order. The upload cell asks for `train/data/train.jsonl` and `train/data/dev.jsonl`
   (make them on your PC first: `python train/make_data.py --n 1500`).
3. Training takes about 1 hour. The last cell downloads `daa-qwen.Q4_K_M.gguf` (~2 GB).
4. On your PC: `python train/install_model.py path\to\daa-qwen.Q4_K_M.gguf`, then measure it:
   `python eval/run_eval.py --model daa-qwen --no-parser`.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
from google.colab import files
if not (os.path.exists("train.jsonl") and os.path.exists("dev.jsonl")):
    files.upload()  # choose train.jsonl and dev.jsonl
print({f: os.path.getsize(f) // 1024 for f in ("train.jsonl", "dev.jsonl")}, "KB")

In [ ]:
import json, torch
from unsloth import FastLanguageModel

MAX_LEN = 6144  # one example (system prompt + 20 tool definitions + question + answer) is ~4.5k tokens
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-3B-Instruct", max_seq_length=MAX_LEN,
                                                     load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=0, use_gradient_checkpointing="unsloth",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

In [ ]:
from datasets import Dataset

def load(path):  # each line already holds the exact prompt text Ollama builds ("text"), made by make_data.py
    return Dataset.from_list([{"text": json.loads(line)["text"]} for line in open(path, encoding="utf-8")])

train, dev = load("train.jsonl"), load("dev.jsonl")
lengths = [len(tokenizer(t)["input_ids"]) for t in train["text"][:100]]
print(len(train), "train,", len(dev), "dev; tokens per example: max", max(lengths), "avg", sum(lengths) // len(lengths))

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train, eval_dataset=dev,
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_LEN, packing=False,
                   per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=1,
                   learning_rate=1e-4, warmup_steps=10, lr_scheduler_type="cosine", weight_decay=0.01,
                   optim="adamw_8bit", logging_steps=10, eval_strategy="steps", eval_steps=60,
                   fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
                   seed=0, output_dir="out", report_to="none"))
# learn only the assistant turns (tool calls and answers), not the prompt or the tool results
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
trainer.train()

Quick check on the dev examples (never trained on): does the model pick the same tool and arguments?

In [ ]:
import re
FastLanguageModel.for_inference(model)
ok = 0
rows = [json.loads(line) for line in open("dev.jsonl", encoding="utf-8")]
for r in rows:
    prompt = r["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = tokenizer.decode(model.generate(**ids, max_new_tokens=200, do_sample=False)[0][ids["input_ids"].shape[1]:])
    gold = r["messages"][2]["tool_calls"][0]["function"]
    m = re.search(r"<tool_call>\s*(\{.*\})\s*</tool_call>", out, re.S)
    try:
        got = json.loads(m.group(1)) if m else None
    except json.JSONDecodeError:
        got = None
    ok += bool(got) and got["name"] == gold["name"] and got.get("arguments") == gold["arguments"]
print(f"dev tool calls exactly right: {ok}/{len(rows)}")

In [ ]:
import glob
model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")  # small backup of the training
model.save_pretrained_gguf("daa-qwen", tokenizer, quantization_method="q4_k_m")
gguf = [g for g in glob.glob("daa-qwen*/*.gguf") + glob.glob("*.gguf") if "Q4_K_M" in g.upper()][0]
os.rename(gguf, "daa-qwen.Q4_K_M.gguf")
files.download("daa-qwen.Q4_K_M.gguf")